In [ ]:
import h5py
import numpy as np
from scipy import stats
import dama as dm
import pandas as pd

from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib import gridspec

In [ ]:
params = {'figure.figsize': (9, 9*0.618),
          'legend.fontsize': 18,
          'axes.labelsize': 18,
          'axes.titlesize': 18,
          'xtick.labelsize': 18,
          'ytick.labelsize': 18}
plt.rcParams.update(params)

path = '/data/sim/IceCubeUpgrade/pisa/level4_queso_v00/' #'/data/user/jweldert/Upgrade_ressources/'

In [ ]:
def load_file(f_name, keys, itype=['all']): #, n_evts=None
    if not type(f_name) == list:
        f_name = [f_name]
    
    var_dict = {}
    for j, fn in enumerate(f_name):
        f = h5py.File(fn, 'r')

        if itype[0] == 'all':
            itype = list(f.keys())

        for k in keys:
            arr = np.array([])
            for i in itype:
                arr = np.append(arr, f[i][k])
            if j == 0:
                var_dict[k] = arr
            else:
                var_dict[k] = np.append(var_dict[k], arr)

        arr = np.array([])
        for i in itype:
            arr = np.append(arr, np.repeat(i, len(f[i]['I3MCWeightDict.weight'])))
        if j == 0:
            var_dict['interaction'] = arr
        else:
            var_dict['interaction'] = np.append(var_dict['interaction'], arr)
    
        f.close()
    
    return dm.PointData(**var_dict)

def calc_baseline_from_coszen(cz, r=6371., h=15., d=1.) :
    '''
    cz = cos(zenith) in radians, to be converted to path length in km
    r = Radius of Earth, in km
    h = Production height in atmosphere, in km
    d = Depth of detector, in km
    '''
    return -r*cz +  np.sqrt( (r*cz)**2 - r**2 + (r+h+d)**2 )

def calc_osc_prob(prob_e, prob_mu, pdg):
    out = np.ones(len(prob_e))
    
    idx = np.where(np.abs(pdg) == 12)[0]
    out[idx] = 1 - prob_e[idx]
    idx = np.where(np.abs(pdg) == 14)[0]
    out[idx] = 1 - prob_mu[idx]
    idx = np.where(np.abs(pdg) == 16)[0]
    out[idx] = prob_e[idx] + prob_mu[idx]
    
    return out

In [ ]:
f = h5py.File(path+"ICU_pisa_genie_028_level4_queso_v00.hdf5", 'r')
print(f.keys())
print('------------------------------------------------------')
print(f['nue_cc'].keys())
f.close()

## Load files

In [ ]:
%%time
f_name = path+"ICU_pisa_genie_028_level4_queso_v00.hdf5"
keys = [
        'MCInIcePrimary.pdg_encoding', 'I3MCWeightDict.prob_from_nue',
        'I3MCWeightDict.prob_from_numu', 'I3MCWeightDict.weight_no_osc', 'I3MCWeightDict.weight',
    
        'MCInIcePrimary.dir.coszen', 'MCInIcePrimary.energy', 'MCInIcePrimary.pos.x', 
        'MCInIcePrimary.pos.y', 'MCInIcePrimary.pos.z', 'MCInIcePrimary.time', 'tau_correction', 
        'true_cascade_energy', 'true_track_energy', 'MCExtraTruthInfo.vertex_rho36', 'MCInIcePrimary.dir.azimuth',
    
        "QuesoL3_Vars_cleaned_num_hit_modules", "QuesoL3_Vars_cleaned_num_hits_fid_vol", "QuesoL3_Bool",
        "QuesoL3_Vars_cleaned_vertexZ", "QuesoL3_Vars_cleaned_length", "QuesoL3_Vars_uncleaned_length",
    
        "QuesoL4_Vars_L4_muon_firstBDT_score", "QuesoL4_Vars_L4_muon_secondBDT_score", "QuesoL4_Vars_graphnet_L4_neutrino_pred",
        "QuesoL4_Bool",
    
        'SplitInIcePulses_dynedge_v2_PulsesUpgradeHitMultiplicity.n_hit_mdoms',
    
        #'FreeDOM_azimuth', 'FreeDOM_coszen', 'FreeDOM_energy', 'FreeDOM_free_fit_llh', 'FreeDOM_n_iters',
        #'FreeDOM_rho36', 'FreeDOM_time', 'FreeDOM_track_energy', 'FreeDOM_track_frac', 'FreeDOM_x', 'FreeDOM_y',
        #'FreeDOM_z',
    
        'graphnet_dynedge_coszen_reconstruction_coszen_pred', 'graphnet_dynedge_energy_reconstruction_energy_pred',
        'graphnet_dynedge_neutrino_classification_neutrino_pred', 'graphnet_dynedge_track_classification_track_pred',
        'graphnet_dynedge_zenith_reconstruction_zenith_kappa',
       ]

MC = load_file(f_name, keys)
MC.baseline = calc_baseline_from_coszen(MC['MCInIcePrimary.dir.coszen'])
MC.osc_prob = calc_osc_prob(MC['I3MCWeightDict.prob_from_nue'], MC['I3MCWeightDict.prob_from_numu'], MC['MCInIcePrimary.pdg_encoding'])
MC.LoE = calc_baseline_from_coszen(MC.graphnet_dynedge_coszen_reconstruction_coszen_pred) / MC.graphnet_dynedge_energy_reconstruction_energy_pred
print(f'Loaded {MC.size} events')

cuts_MC = (MC['I3MCWeightDict.weight'] > 0) & (MC.QuesoL4_Bool == 1) #& (MC.graphnet_dynedge_coszen_reconstruction_coszen_pred < 0.3) & (MC.osc_prob >= 0.1) 
#cuts_MC = cuts_MC & (MC.FreeDOM_n_iters > 0) & (MC.FreeDOM_rho36 < 290) & (MC.FreeDOM_energy < 400) & (MC.FreeDOM_coszen < 0.3)
np.sum(cuts_MC)

In [ ]:
%%time
f_name = path+"ICU_pisa_muongun_028_level4_queso_v00.hdf5"
keys = [
        'I3EventHeader.run_id', 'I3EventHeader.sub_run_id', 'I3MCWeightDict.weight', 'MCInIcePrimary.dir.coszen',
        'MCInIcePrimary.energy', 'MCInIcePrimary.pos.x', 'MCInIcePrimary.pos.y', 'MCInIcePrimary.pos.z', 'data_livetime', 

        "QuesoL3_Vars_cleaned_num_hit_modules", "QuesoL3_Vars_cleaned_num_hits_fid_vol", "QuesoL3_Bool",
        "QuesoL3_Vars_cleaned_vertexZ", "QuesoL3_Vars_cleaned_length", "QuesoL3_Vars_uncleaned_length",
    
        "QuesoL4_Vars_L4_muon_firstBDT_score", "QuesoL4_Vars_L4_muon_secondBDT_score", "QuesoL4_Vars_graphnet_L4_neutrino_pred",
        "QuesoL4_Bool",
    
        #'FreeDOM_azimuth', 'FreeDOM_coszen', 'FreeDOM_energy', 'FreeDOM_free_fit_llh', 'FreeDOM_n_iters',
        #'FreeDOM_rho36', 'FreeDOM_time', 'FreeDOM_track_energy', 'FreeDOM_track_frac', 'FreeDOM_x', 'FreeDOM_y',
        #'FreeDOM_z',
    
        'graphnet_dynedge_coszen_reconstruction_coszen_pred', 'graphnet_dynedge_energy_reconstruction_energy_pred',
        'graphnet_dynedge_neutrino_classification_neutrino_pred', 'graphnet_dynedge_track_classification_track_pred',
        'graphnet_dynedge_zenith_reconstruction_zenith_kappa',
       ]

muon = load_file(f_name, keys)
print(f'Loaded {muon.size} events')

cuts_muon = (muon['I3MCWeightDict.weight'] > 0) & (muon.QuesoL4_Bool == 1) #& (muon.graphnet_dynedge_coszen_reconstruction_coszen_pred < 0.3)
#cuts_muon = cuts_muon & (muon.FreeDOM_n_iters > 0) & (muon.FreeDOM_rho36 < 290) & (muon.FreeDOM_energy < 400) & (muon.FreeDOM_coszen < 0.3)
np.sum(cuts_muon)

In [ ]:
%%time
f_name = path+"ICU_pisa_noise_028_level4_queso_v00.hdf5"
keys = [
        'I3MCWeightDict.weight',

        "QuesoL3_Vars_cleaned_num_hit_modules", "QuesoL3_Vars_cleaned_num_hits_fid_vol", "QuesoL3_Bool",
        "QuesoL3_Vars_cleaned_vertexZ", "QuesoL3_Vars_cleaned_length", "QuesoL3_Vars_uncleaned_length",
    
        "QuesoL4_Vars_L4_muon_firstBDT_score", "QuesoL4_Vars_L4_muon_secondBDT_score", "QuesoL4_Vars_graphnet_L4_neutrino_pred",
        "QuesoL4_Bool",
    
        #'FreeDOM_azimuth', 'FreeDOM_coszen', 'FreeDOM_energy', 'FreeDOM_free_fit_llh', 'FreeDOM_n_iters',
        #'FreeDOM_rho36', 'FreeDOM_time', 'FreeDOM_track_energy', 'FreeDOM_track_frac', 'FreeDOM_x', 'FreeDOM_y',
        #'FreeDOM_z',
    
        'graphnet_dynedge_coszen_reconstruction_coszen_pred', 'graphnet_dynedge_energy_reconstruction_energy_pred',
        'graphnet_dynedge_neutrino_classification_neutrino_pred', 'graphnet_dynedge_track_classification_track_pred',
        'graphnet_dynedge_zenith_reconstruction_zenith_kappa',
       ]

noise = load_file(f_name, keys)
print(f'Loaded {noise.size} events')

cuts = (noise.QuesoL4_Bool == 1)
np.sum(cuts), np.sum(noise['I3MCWeightDict.weight'][cuts])*1e9

## Plot

In [ ]:
#var, bins, name = 'MCInIcePrimary.energy', np.logspace(np.log10(1),np.log10(500),12), r'$E^{deposited}_{true}$'
#var, bins, name = 'FreeDOM_energy', np.logspace(np.log10(3),np.log10(300),12), r'$E^{deposited}_{reco}$'
#var, bins, name = 'graphnet_dynedge_energy_reconstruction_energy_pred', np.logspace(np.log10(3),np.log10(300),30), r'$E^{deposited}_{reco}$'
#var, bins, name = 'num_hit_modules', np.linspace(4,100,49), '#hit modules'
#var, bins, name = 'FreeDOM_coszen', np.linspace(-1,0.3,10), r'$\cos(\theta^{zenith})_{reco}$'
var, bins, name = 'graphnet_dynedge_coszen_reconstruction_coszen_pred', np.linspace(-1,0.3,30), r'$\cos(\theta^{zenith})_{reco}$'
#var, bins, name = 'graphnet_dynedge_track_classification_track_pred', np.linspace(0,1,15), r'PID score'
#var, bins, name = 'graphnet_dynedge_neutrino_classification_neutrino_pred', np.linspace(0.95,1,15), r'neutrino score'
#var, bins, name = 'MCInIcePrimary.dir.coszen', np.linspace(-1,0.3,30), r'$\cos(\theta^{zenith})_{true}$'
#var, bins, name = 'LoE', np.logspace(0,3.7,50), 'L/E'

cc = (MC['interaction']=='nue_cc') | (MC['interaction']=='nuebar_cc')
nu_e_counts, bins = np.histogram(MC[var][(cuts_MC) & (cc)], bins, weights=MC['I3MCWeightDict.weight'][(cuts_MC) & (cc)]) #
cc = (MC['interaction']=='numu_cc') | (MC['interaction']=='numubar_cc')
nu_mu_counts, bins = np.histogram(MC[var][(cuts_MC) & (cc)], bins, weights=MC['I3MCWeightDict.weight'][(cuts_MC) & (cc)]) #
cc = (MC['interaction']=='nutau_cc') | (MC['interaction']=='nutaubar_cc')
nu_tau_counts, bins = np.histogram(MC[var][(cuts_MC) & (cc)], bins, weights=MC['I3MCWeightDict.weight'][(cuts_MC) & (cc)]) #
nc = (MC['interaction']=='nue_nc') | (MC['interaction']=='numu_nc') | (MC['interaction']=='nutau_nc')
nc = nc | (MC['interaction']=='nue_nc') | (MC['interaction']=='numu_nc') | (MC['interaction']=='nutau_nc')
nu_nc_counts, bins = np.histogram(MC[var][(cuts_MC) & (nc)], bins, weights=MC['I3MCWeightDict.weight'][(cuts_MC) & (nc)]) #

muon_counts, bins = np.histogram(muon[var][cuts_muon], bins, weights=muon['I3MCWeightDict.weight'][cuts_muon])

In [ ]:
plt.bar((bins[:-1]+bins[1:])/2, nu_e_counts, (bins[1:]-bins[:-1]), label=r'$\nu_{e}^{CC}$', color='red')
plt.bar((bins[:-1]+bins[1:])/2, nu_mu_counts, (bins[1:]-bins[:-1]), bottom=nu_e_counts, label=r'$\nu_{\mu}^{CC}$', color='blue')
plt.bar((bins[:-1]+bins[1:])/2, nu_tau_counts, (bins[1:]-bins[:-1]), bottom=(nu_e_counts+nu_mu_counts), label=r'$\nu_{\tau}^{CC}$', color='green')
plt.bar((bins[:-1]+bins[1:])/2, nu_nc_counts, (bins[1:]-bins[:-1]), bottom=(nu_e_counts+nu_mu_counts+nu_tau_counts), label=r'$\nu^{NC}$', color='orange')
plt.bar((bins[:-1]+bins[1:])/2, muon_counts, (bins[1:]-bins[:-1]), bottom=(nu_nc_counts+nu_e_counts+nu_mu_counts+nu_tau_counts), label=r'$\mu_{atm}$', alpha=0.5)

#plt.hist(MC[var][(cuts_MC)], bins, weights=MC['I3MCWeightDict.weight_no_osc'][(cuts_MC)], histtype='step', linestyle='--', color='grey', label='no osc')

plt.title('Upgrade', size=20)
if not np.isclose(np.diff(bins)[0], np.diff(bins)[1]): plt.xscale('log')
plt.ylabel('Rate [1/s]', size=18)
plt.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
plt.xticks(fontsize=18)
plt.xlabel(name)
plt.xlim(bins[0], bins[-1])
plt.yticks(fontsize=18)
#plt.ylim(0,9e-3)
#plt.yscale('log')
plt.legend(prop={'size':16})

#plt.savefig('../plots/dynedge_coszen', bbox_inches='tight')

livetime

In [ ]:
bins = np.logspace(0,2.5,30)
which_bin = np.digitize(MC['MCInIcePrimary.energy'], bins)

livetimes = [[], [], [], [], [], []]
for i in range(len(bins)):
    mask = (MC.weights > 0) & (which_bin == i) & ((MC.interaction == 'nue_cc') | (MC.interaction == 'nue_nc'))
    livetimes[0].append(np.sum(MC['I3MCWeightDict.weight'][mask])/np.sum(np.square(MC['I3MCWeightDict.weight'][mask])))
    
    mask = (MC.weights > 0) & (which_bin == i) & ((MC.interaction == 'numu_cc') | (MC.interaction == 'numu_nc'))
    livetimes[1].append(np.sum(MC['I3MCWeightDict.weight'][mask])/np.sum(np.square(MC['I3MCWeightDict.weight'][mask])))
    
    mask = (MC.weights > 0) & (which_bin == i) & ((MC.interaction == 'nutau_cc') | (MC.interaction == 'nutau_nc'))
    livetimes[2].append(np.sum(MC['I3MCWeightDict.weight'][mask])/np.sum(np.square(MC['I3MCWeightDict.weight'][mask])))
    
    mask = (MC.weights > 0) & (which_bin == i) & ((MC.interaction == 'nuebar_cc') | (MC.interaction == 'nuebar_nc'))
    livetimes[3].append(np.sum(MC['I3MCWeightDict.weight'][mask])/np.sum(np.square(MC['I3MCWeightDict.weight'][mask])))
    
    mask = (MC.weights > 0) & (which_bin == i) & ((MC.interaction == 'numubar_cc') | (MC.interaction == 'numubar_nc'))
    livetimes[4].append(np.sum(MC['I3MCWeightDict.weight'][mask])/np.sum(np.square(MC['I3MCWeightDict.weight'][mask])))
    
    mask = (MC.weights > 0) & (which_bin == i) & ((MC.interaction == 'nutaubar_cc') | (MC.interaction == 'nutaubar_nc'))
    livetimes[5].append(np.sum(MC['I3MCWeightDict.weight'][mask])/np.sum(np.square(MC['I3MCWeightDict.weight'][mask])))
    
livetimes= np.array(livetimes)

In [ ]:
labels = [r'$\nu_{e}$', r'$\nu_{\mu}$', r'$\nu_{\tau}$', r'$\bar{\nu}_{e}$', r'$\bar{\nu}_{\mu}$', r'$\bar{\nu}_{\tau}$']

plt.figure(figsize=(5,3))
for i in range(len(livetimes)):
    plt.plot(bins, livetimes[i]/(60*60*24*365), linewidth=2, label=labels[i])
    
plt.legend(prop={'size':13})
plt.title('Neutrinos')
plt.loglog()
plt.xlabel('True energy (GeV)')
plt.ylim(0.3,1000)
plt.ylabel('livetime (years)')

#plt.savefig('/data/user/jweldert/notebooks/plots/genie_sim/genie_livetime', bbox_inches='tight')

## Add variables to hdf5 file

Add GENIE systematics

In [ ]:
def fit_genie_syst(in_yvalues, genie_weight):
    rw_xvalues  = np.array([-2, -1, 0, 1, 2])
    if sum(in_yvalues) == 4.0:
        return np.array([0, 0])
    in_yvalues = np.array(in_yvalues)
    yvalues = np.concatenate(
        (in_yvalues[:2]/genie_weight, [1.], in_yvalues[2:]/genie_weight)
    )
    # returns  the coefficients from low to high order (as opposed to np.polyfit)
    fitcoeff = np.polynomial.polynomial.polyfit(
        rw_xvalues, yvalues, deg=2
    )
    linear_fit_coefft, quadratic_fit_coefft = fitcoeff[1:]
    return linear_fit_coefft, quadratic_fit_coefft

def calc_genie_systematics(group):
    '''
    Calculate GENIE systematics for a single group
    '''
    genie_weight = np.array(group['I3MCWeightDict.GENIEWeight'])
    
    for genie_syst in ['MaCCQE', 'MaCCRES']:
        # same polynomial fits to all GENIE systematics
        linear_fit = np.zeros(len(genie_weight))
        quad_fit = np.zeros(len(genie_weight))

        # Get the information pertainting to the GENIE axial mass
        # systematics that have been written to the hdf5 file
        genie_rw_syst = []
        # collect all weight variations for this systematic
        for i in (0, 1, 2, 3):
            genie_rw_syst_i = group['I3GenieSystWeightDict.rw_'+genie_syst+'.%d' % i]
            genie_rw_syst.append(genie_rw_syst_i)

        genie_rw_syst = np.array(genie_rw_syst)
        
        # obtain linear and quadratic fit coefficients on an
        # event-by-event basis
        for event_id, gw in enumerate(genie_weight):
            rw_syst_list = genie_rw_syst[:, event_id]
            
            # get the linear and quadratic terms from a poly fit
            linear_fit[event_id], quad_fit[event_id] = fit_genie_syst(
                in_yvalues=rw_syst_list,
                genie_weight=gw
            )
        
        # add the fit coefficients to the events
        #del(group['linear_fit_%s' % genie_syst])
        group['linear_fit_%s' % genie_syst] = linear_fit
        #del(group['quad_fit_%s' % genie_syst])
        group['quad_fit_%s' % genie_syst] = quad_fit

In [ ]:
%%time
f = h5py.File("/data/sim/IceCubeUpgrade/pisa/ICU_pisa_genie.hdf5", 'r+')
for k in f.keys():
    calc_genie_systematics(f[k])
f.close()

correct weights

In [ ]:
%%time
f = h5py.File("/data/sim/IceCubeUpgrade/pisa/ICU_pisa_genie.hdf5", 'r+')
for k in f.keys():
    w = np.array(f[k]['I3MCWeightDict.weight']) * np.array(f[k]['I3MCWeightDict.NEvents']) / np.array(f[k]['I3GenieInfo.n_flux_events'])
    del(f[k]['I3MCWeightDict.weight'])
    f[k]['I3MCWeightDict.weight'] = w
    
    w = np.array(f[k]['I3MCWeightDict.weight_no_osc']) * np.array(f[k]['I3MCWeightDict.NEvents']) / np.array(f[k]['I3GenieInfo.n_flux_events'])
    del(f[k]['I3MCWeightDict.weight_no_osc'])
    f[k]['I3MCWeightDict.weight_no_osc'] = w
f.close()

freedom stuff

In [ ]:
def get_indices(x, y):
    isin = np.isin(x, y)
    xi = x[isin]
    
    idxs = np.argsort(y)
    xe = np.searchsorted(y[idxs], xi)
    indices = idxs[xe]
    
    assert np.all(y[indices] == xi)
    return indices, isin

In [ ]:
%%time
f = h5py.File("/data/user/jweldert/Upgrade_ressources/ICU_pisa_genie_0028.hdf5", 'r+')
#f = h5py.File("/data/user/jweldert/Upgrade_ressources/ICU_pisa_muongun_0028.hdf5", 'r+')
for k in f.keys():
    if 'numu' in k:
        K = k.split('_')[0]
    elif 'nue' in k:
        K = 'nue'
    elif 'nutau' in k:
        K = 'nutau'
    else:
        K = k

    recos = pd.read_pickle('/data/user/jweldert/Upgrade_results/'+K+'_reco.pkl')
    ind, pos = get_indices((np.array(f[k]['MCInIcePrimary.pos.x']).astype(np.float32)*10000).astype(int)+
                           (np.array(f[k]['MCInIcePrimary.pos.y']).astype(np.float32)*100000000).astype(int)+
                           (np.array(f[k]['MCInIcePrimary.pos.z']).astype(np.float32)*1000000000000).astype(int)+
                           (np.array(f[k]['num_hit_modules_no_PDOM'])*10000000000000).astype(int), 
                           (np.array(recos.x_true)*10000).astype(int)+
                           (np.array(recos.y_true)*100000000).astype(int)+
                           (np.array(recos.z_true)*1000000000000).astype(int)+
                           (np.array(recos.nCh)*10000000000000).astype(int)
                          )
    arr = np.zeros(len(pos))
    
    arr[pos] = np.array(recos.total_energy[ind])
    f[k]['FreeDOM_energy'] = arr
    arr[pos] = np.cos(recos.zenith[ind])
    f[k]['FreeDOM_coszen'] = arr
    arr[pos] = np.array(recos.x[ind])
    f[k]['FreeDOM_x'] = arr
    arr[pos] = np.array(recos.y[ind])
    f[k]['FreeDOM_y'] = arr
    arr[pos] = np.array(recos.z[ind])
    f[k]['FreeDOM_z'] = arr
    arr[pos] = np.array(recos.time[ind])
    f[k]['FreeDOM_time'] = arr
    arr[pos] = np.array(recos.azimuth[ind])
    f[k]['FreeDOM_azimuth'] = arr
    arr[pos] = np.array(recos.track_frac[ind])
    f[k]['FreeDOM_track_frac'] = arr
    arr[pos] = np.array(recos.free_fit_llh[ind])
    f[k]['FreeDOM_free_fit_llh'] = arr
    arr[pos] = np.array(recos.n_iters[ind])
    f[k]['FreeDOM_n_iters'] = arr
    #f[k]['FreeDOM_n_calls'] = np.array(recos.n_calls[ind])
    #f[k]['FreeDOM_reco_time'] = np.array(recos.reco_time[ind])
    
    #unc?
    
    #recos = pd.read_pickle('/data/user/jweldert/Upgrade_results/'+K+'_reco_no_track.pkl') #new ind?
    #f[k]['FreeDOM_no_track_free_fit_llh'] = np.array(recos.free_fit_llh[ind])

f.close()

In [ ]:
%%time
f = h5py.File("/data/user/jweldert/Upgrade_ressources/ICU_pisa_genie_0028.hdf5", 'r+')
#f = h5py.File("/data/user/jweldert/Upgrade_ressources/ICU_pisa_muongun_0028.hdf5", 'r+')
for k in f.keys():
    f[k]['FreeDOM_rho36'] = np.sqrt((np.array(f[k]['FreeDOM_x'])-46.29)**2 + (np.array(f[k]['FreeDOM_y'])+34.88)**2)
    #f[k]['FreeDOM_gof'] = -f[k]['FreeDOM_free_fit_llh'] / f[k]['n_pulses']
    f[k]['FreeDOM_track_energy'] = np.array(f[k]['FreeDOM_energy']) * np.array(f[k]['FreeDOM_track_frac'])
    #del(f[k]['FreeDOM_pid'])
    #f[k]['FreeDOM_pid'] = np.abs(f[k]['FreeDOM_no_track_free_fit_llh'] - f[k]['FreeDOM_free_fit_llh']) / f[k]['Qtot']
f.close()